In [27]:
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

In [18]:
from tensorflow.keras import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Input

def cria_modelo_com_sequencial_api():
    seq_model = Sequential([Flatten(input_shape=(28, 28)),
                            Dense(128, activation=tf.nn.relu),
                            Dense(10, activation='softmax')
                          ])

    return seq_model

In [19]:
def cria_modelo_com_functional_api():
    input_layer   = Input(shape=(28, 28))

    # empilha as camadas usando: new_layer()(previous_layer)
    flatten_layer = Flatten()(input_layer)
    first_dense   = Dense(128, activation='relu')(flatten_layer)
    output_layer  = Dense(10, activation='softmax')(first_dense)

    # declara inputs e outputs
    func_model = Model(inputs=input_layer, outputs=output_layer)

    return func_model

In [20]:
#carrega dataset
mnist = tf.keras.datasets.fashion_mnist
(training_images, training_labels), (test_images, test_labels) = mnist.load_data()

#normaliza dados
training_images = training_images / 255.0
test_images     = test_images / 255.0

#configura modelo
modelo = cria_modelo_com_functional_api()
modelo.compile(optimizer=tf.optimizers.Adam(),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

#treina modelo
history = modelo.fit(training_images, training_labels, epochs=20)

Epoch 1/20
 570/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - accuracy: 0.7870 - loss: 0.6194

KeyboardInterrupt: 

In [21]:
data_url_red   = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
data_url_white = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'

# Carregar o dataset
data_red             = pd.read_csv(data_url_red, delimiter=';')
data_red['classe']   = 1 
data_white           = pd.read_csv(data_url_white, delimiter=';')
data_white['classe'] = 0
data                 = pd.concat([data_red, data_white])
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,classe
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,1
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,1
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,0
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,0
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,0
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,0


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separar as entradas e saídas
y     = data['quality']
y_bin = data['classe']

X = data.drop(columns=['quality', 'classe'])


# Dividir em conjunto de treino e teste
X_train, X_test, y_train, y_test, y_bin_train, y_bin_test = train_test_split(X, y, y_bin, test_size=0.2, random_state=42)

# Normalizar os dados
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [23]:
def cria_modelo_multiplas_saidas():
    entrada = Input(shape=(11, ))

    dense_1 = Dense(128, activation='relu')(entrada)
    dense_2 = Dense(64, activation='relu')(dense_1)

    saida_1 = Dense(1, name='qualidade')(dense_2)

    dense_3 = Dense(64, activation='relu')(dense_2)
    saida_2 = Dense(1, activation='sigmoid', name='classe')(dense_3)

    modelo_2_saidas = Model(inputs=entrada, outputs=[saida_1, saida_2])

    return modelo_2_saidas

In [24]:
modelo.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 28, 28)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305,312 (1.16 MB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 203,542 (795.09 KB)

In [25]:
modelo.compile(
    optimizer='adam',
    loss={
        'qualidade': 'mse',
        'classe': 'binary_crossentropy'
    },
    metrics={
        'qualidade': 'mae',
        'classe': 'accuracy'
    }
)

In [26]:
history = modelo.fit(
    X_train, {'qualidade': y_train, 'classe': y_bin_train},
    epochs=10,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/10


ValueError: Input 0 with name 'input_layer_1' of layer 'functional_1' is incompatible with the layer: expected shape=(None, 28, 28), found shape=(None, 11)

In [ ]:
def calcula_loss(model, X, y_quality, y_class):
    mse_loss = tf.keras.losses.MeanSquaredError()
    binary_crossentropy_loss = tf.keras.losses.BinaryCrossentropy()

    predictions    = modelo(X)

    qualidade_loss = mse_loss(y_quality, predictions[0])
    classe_loss    = binary_crossentropy_loss(y_class, predictions[1])
    total_loss     = qualidade_loss + classe_loss

    return total_loss

In [ ]:
def treina_modelo(modelo, X, y_qualidade, y_classe, optimizer, train_mae, train_accuracy):
    with tf.GradientTape() as tape:
        loss = calcula_loss(modelo, X, y_qualidade, y_classe)

    # Calcular os gradientes
    gradients = tape.gradient(loss, modelo.trainable_variables)

    # Atualizar os pesos
    optimizer.apply_gradients(zip(gradients, modelo.trainable_variables))

    #predict
    predictions = modelo.predict(X, verbose=0)

    train_mae.update_state(y_qualidade, predictions[0])
    train_accuracy.update_state(y_classe, predictions[1])

    return loss

In [ ]:
modelo         = cria_modelo_multiplas_saidas()
optimizer      = tf.keras.optimizers.Adam()
train_mae      = tf.keras.metrics.MeanAbsoluteError()
train_accuracy = tf.keras.metrics.BinaryAccuracy()

losses, accs, maes = [], [], []

epochs = 100
for epoch in range(epochs):
    train_mae.reset_state()
    train_accuracy.reset_state()

    loss = treina_modelo(modelo, X_train, y_train, y_bin_train, optimizer, train_mae, train_accuracy)
    losses.append(loss.numpy())
    accs.append(train_accuracy.result().numpy())
    maes.append(train_mae.result().numpy())
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.numpy()}, MAE {train_mae.result().numpy()}, Accuracy {train_accuracy.result().numpy()}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 3))
axs      = axs.flatten()

axs[0].plot(losses, label='Loss', color='b')
axs[0].set_title('Loss')

axs[1].plot(accs, label='Acurácia', color='r')
axs[1].set_title('Acurácia')

axs[2].plot(maes, label='MAE', color='k')
axs[2].set_title('Loss')


for i in range(3):
  axs[i].grid(True, linestyle=':')
  axs[i].legend()